In [1]:
import os
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [10]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [11]:
os.makedirs("data", exist_ok=True)

sample_text = """
Data analysis is the process of collecting, cleaning, exploring, and interpreting data to find useful insights.

RAG means Retrieval-Augmented Generation. It helps an AI model answer questions using external documents.

Generative AI can create text, code, summaries, and answers based on user prompts.

Embeddings convert text into numerical vectors so similar meanings can be searched.

ChromaDB is a vector database used to store and retrieve document embeddings.
"""

with open("data/sample_notes.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

In [12]:
loader = TextLoader("data/sample_notes.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3
)

In [14]:
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Use the document context below to answer the question.
If the context is not useful, say exactly: NOT_IN_CONTEXT

Context:
{context}

Question:
{question}

Answer:
""")

general_prompt = ChatPromptTemplate.from_template("""
You are a helpful mini AI assistant.

Answer the user's question clearly and simply.

Question:
{question}

Answer:
""")

In [15]:
def hybrid_ai_assistant(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    rag_chain = rag_prompt | llm

    rag_response = rag_chain.invoke({
        "context": context,
        "question": question
    }).content

    if "NOT_IN_CONTEXT" not in rag_response:
        return {
            "mode": "RAG Mode",
            "answer": rag_response
        }

    general_chain = general_prompt | llm

    general_response = general_chain.invoke({
        "question": question
    }).content

    return {
        "mode": "General LLM Mode",
        "answer": general_response
    }

In [16]:
response = hybrid_ai_assistant("How does RAG and LLM work?")
print("Mode:", response["mode"])
print(response["answer"])

Mode: General LLM Mode
RAG (Retrieve, Augment, Generate) and LLM (Large Language Model) are related technologies that work together to generate human-like text. 

Here's a simplified explanation of how they work:

1. **LLM (Large Language Model)**: This is a type of artificial intelligence trained on vast amounts of text data. It learns patterns and relationships in language, allowing it to predict and generate text.

2. **RAG (Retrieve, Augment, Generate)**: RAG is a framework that uses LLM to generate text. It works in three steps:
   - **Retrieve**: It searches a database or knowledge base to find relevant information related to the input or question.
   - **Augment**: It uses the retrieved information to create a context or prompt for the LLM.
   - **Generate**: The LLM uses this context to generate a response or answer.

In essence, RAG helps LLM by providing it with relevant information to generate more accurate and informative responses. This combination enables the model to pro